# 02 · Generate demand forecast (Step 3 — prerequisite for the Demand Twin)

Builds a forward-looking demand forecast per **material × plant × distribution channel**
from sales history and writes it in the canonical `demandplan` shape. The Demand Twin and
all downstream planning consume this.

**Why first:** the forecast target table often exists but is empty (true for CoffCoff's
`gold.demandplan`). Nothing demand-driven can be built until it is populated.

**Approach:** a distributed, **pluggable** per-series model run via `applyInPandas`. The
default model is **`statsforecast` `AutoETS`** (prediction intervals that widen with the
horizon), with a dependency-free **`SeasonalNaiveMA`** fallback. We deliberately do **not**
use `ai_forecast` or AutoML.

**Output:** the forecast is written directly to **`gold.demandplan`** — the demand plan is a
canonical planning input, so this is the one documented exception to gold being read-only.
Backtest accuracy goes to `monitoring.demand_forecast_accuracy`.

**Quality:** before writing, a **rolling-origin backtest** scores each series (WMAPE, MAE,
bias, PI coverage); results are displayed and written to `monitoring.demand_forecast_accuracy`.

_Runs on serverless: no `.cache()`/`.persist()` (unsupported there)._

In [0]:
# statsforecast powers the default AutoETS model. Pin a version for reproducibility in prod.
%pip install -q statsforecast

## Config

In [0]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
# By design, the forecast populates gold.demandplan (the demand plan is a canonical planning
# input) — the ONE documented exception to "gold is read-only". Blank = gold.demandplan; set
# explicitly to write elsewhere (e.g. mickmickstableaws_catalog.monitoring.demand_plan).
dbutils.widgets.text("forecast_output_table", "", "Forecast output table (blank = gold.demandplan)")
dbutils.widgets.dropdown("model", "AutoETS", ["AutoETS", "SeasonalNaiveMA"], "Forecast model")
dbutils.widgets.text("granularity", "W", "Bucket: D | W | M")
dbutils.widgets.text("horizon_steps", "13", "Forecast horizon (buckets)")
dbutils.widgets.text("history_buckets", "0", "History lookback (buckets; 0 = all)")
dbutils.widgets.text("backtest_windows", "6", "Rolling-origin backtest windows (0 = skip)")
dbutils.widgets.text("demand_date_basis", "ORDER_DATE", "ORDER_DATE | REQUESTED_DELIVERY_DATE")
dbutils.widgets.text("confidence_level", "0.95", "Prediction-interval confidence")

import uuid
from datetime import datetime

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
OUT_TABLE = dbutils.widgets.get("forecast_output_table").strip() or f"{SRC}.demandplan"
ACCURACY_TABLE = f"{FQ_MON}.demand_forecast_accuracy"
MODEL = dbutils.widgets.get("model").strip()
GRANULARITY = dbutils.widgets.get("granularity").strip().upper()
HORIZON = int(dbutils.widgets.get("horizon_steps"))
HISTORY = int(dbutils.widgets.get("history_buckets"))
BACKTEST_WINDOWS = int(dbutils.widgets.get("backtest_windows"))
DEMAND_BASIS = dbutils.widgets.get("demand_date_basis").strip().upper()
CONF = float(dbutils.widgets.get("confidence_level"))

SEASON = {"D": 7, "W": 52, "M": 12}[GRANULARITY]
MIN_TRAIN = max(2 * SEASON, 8)   # need >= 2 seasonal cycles to validate a seasonal model

# Shared version ids for this run (forecast rows + accuracy rows share them).
FORECAST_RUN_ID = str(uuid.uuid4())
FORECAST_VERSION = f"{datetime.utcnow():%Y%m%d%H%M%S}_{FORECAST_RUN_ID[:8]}"

# --- Source column mapping (CoffCoff salesorders defaults). EXTEND for your business. ---
SALES_TABLE   = f"{SRC}.salesorders"
COL_MANDT     = "Client_MANDT"
COL_MATERIAL  = "MaterialNumber_MATNR"
COL_PLANT     = "Plant_WERKS"
COL_CHANNEL   = "DistributionChannel_VTWEG"
COL_QTY       = "CumulativeOrderQuantity_KWMENG"
COL_UOM       = "SalesUnit_VRKME"
COL_ORDER_DT  = "CreationDate_ERDAT"
COL_REQ_DT    = "RequestedDeliveryDate_VDATU"
DEMAND_DATE_COL = COL_ORDER_DT if DEMAND_BASIS == "ORDER_DATE" else COL_REQ_DT

print(f"Source: {SALES_TABLE}\nOutput: {OUT_TABLE}\nModel: {MODEL}  Grain: {GRANULARITY}  "
      f"Horizon: {HORIZON}  History: {'all' if HISTORY==0 else HISTORY}  Backtest: {BACKTEST_WINDOWS}")

## Build the history series (material × plant × channel × period)
Aggregate sales into periodic buckets, then optionally keep only the last `history_buckets`
periods per series (`0` = use all history). Materialized to a temp view that the model and
backtest read from.

In [0]:
from pyspark.sql import Window, functions as F

trunc = {"D": "day", "W": "week", "M": "month"}[GRANULARITY]

history = spark.sql(f"""
SELECT
  `{COL_MANDT}`     AS mandt,
  `{COL_MATERIAL}`  AS material,
  `{COL_PLANT}`     AS plant,
  `{COL_CHANNEL}`   AS channel,
  MAX(`{COL_UOM}`)  AS uom,
  date_trunc('{trunc}', `{DEMAND_DATE_COL}`) AS period_start,
  SUM(CAST(`{COL_QTY}` AS DOUBLE))           AS qty
FROM {SALES_TABLE}
WHERE `{DEMAND_DATE_COL}` IS NOT NULL
  AND `{COL_MATERIAL}` IS NOT NULL
GROUP BY 1,2,3,4,6
""")

# Wire history_buckets: keep only the most recent HISTORY periods per series (0 = all).
if HISTORY > 0:
    w = Window.partitionBy("material", "plant", "channel").orderBy(F.col("period_start").desc())
    history = history.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") <= HISTORY).drop("_rn")

history.createOrReplaceTempView("history")
forecast_origin = spark.table("history").selectExpr("max(period_start) AS mx").collect()[0]["mx"]
print(f"History rows: {spark.table('history').count():,}  Forecast origin: {forecast_origin}")

## The model (pluggable)
`model_forecast(y, h, ...)` returns `h` future means with lower/upper prediction bounds.
Default `AutoETS` (statsforecast) gives intervals that widen with the horizon; the
dependency-free `SeasonalNaiveMA` widens its interval as `sigma * sqrt(step)`. Both floor
forecasts at zero. Per series, `AutoETS` failures fall back to the naive model.

`# TODO — EXTEND for your business:` add another branch (e.g. Prophet, AutoARIMA).

In [0]:
import numpy as np
from scipy.stats import norm  # installed with statsforecast; falls back to Z=1.96 if missing

try:
    Z = float(norm.ppf(0.5 + CONF / 2.0))
except Exception:
    Z = 1.96  # ~95%

LEVEL_PCT = int(round(CONF * 100))


def _seasonal_naive_ma(y, h, season, z):
    n = len(y)
    k = min(8, n)
    level = float(np.mean(y[-k:])) if k > 0 else 0.0
    sigma = float(np.std(y[-min(season, n):])) if n > 1 else max(level * 0.25, 1.0)
    mean, lo, hi = [], [], []
    for step in range(1, h + 1):
        seas = 1.0
        if n >= season:
            base = float(np.mean(y[-season:])) or 1.0
            seas = float(y[-season + (step - 1) % season]) / base if base else 1.0
            seas = min(max(seas, 0.5), 2.0)
        fc = max(level * seas, 0.0)
        s = sigma * np.sqrt(step)          # widen the interval with horizon
        mean.append(fc); lo.append(max(fc - z * s, 0.0)); hi.append(fc + z * s)
    return np.array(mean), np.array(lo), np.array(hi)


def _auto_ets(y, h, season, level_pct):
    from statsforecast.models import AutoETS
    sl = season if len(y) >= 2 * season else 1     # seasonal only with >= 2 cycles
    res = AutoETS(season_length=sl).forecast(y=y.astype(float), h=h, level=[level_pct])
    mean = np.clip(res["mean"], 0.0, None)
    lo = np.clip(res[f"lo-{level_pct}"], 0.0, None)
    hi = res[f"hi-{level_pct}"]
    return mean, lo, hi


def model_forecast(y, h, season, model_name, z, level_pct):
    y = np.asarray(y, dtype=float)
    if model_name == "AutoETS" and len(y) >= 4:
        try:
            return _auto_ets(y, h, season, level_pct)
        except Exception:
            pass                                    # graceful per-series fallback
    return _seasonal_naive_ma(y, h, season, z)

## Rolling-origin backtest
Expanding-window validation: for each of `backtest_windows` non-overlapping `HORIZON`-length
blocks at the end of a series, train on the prior history and score the forecast. Metrics are
volume-weighted: **WMAPE**, **MAE**, **bias** (signed), and **PI coverage** (share of actuals
inside the interval; should be near the confidence level). Series with `< MIN_TRAIN` history
are skipped. Per-series results (incl. raw error sums for exact weighting) are written to
`monitoring.demand_forecast_accuracy`, then read back for the summary.

In [0]:
import pandas as pd

BT_SCHEMA = ("material string, plant string, channel string, points_scored int, "
             "abs_err_sum double, act_sum double, signed_sum double, wmape double, "
             "mae double, bias double, pi_coverage double, model_used string")


def _backtest(y, h, season, model_name, z, level_pct, n_windows, min_train, step):
    n = len(y)
    abs_err = act = signed = cov = tot = 0.0
    for wd in range(n_windows):
        split = n - (n_windows - wd) * step
        if split < min_train:
            continue
        train, test = y[:split], y[split:split + h]
        if len(test) == 0:
            continue
        mean, lo, hi = model_forecast(train, len(test), season, model_name, z, level_pct)
        err = mean - test
        abs_err += float(np.sum(np.abs(err)))
        act += float(np.sum(np.abs(test)))
        signed += float(np.sum(err))
        cov += float(np.sum((test >= lo) & (test <= hi)))
        tot += len(test)
    if tot == 0:
        return None
    return (int(tot), abs_err, act, signed,
            (abs_err / act if act > 0 else None),
            (abs_err / tot),
            (signed / act if act > 0 else None),
            (cov / tot))


def backtest_group(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("period_start")
    y = pdf["qty"].astype(float).to_numpy()
    res = _backtest(y, HORIZON, SEASON, MODEL, Z, LEVEL_PCT, BACKTEST_WINDOWS, MIN_TRAIN, HORIZON)
    cols = [c.split()[0] for c in BT_SCHEMA.split(", ")]
    if res is None:
        return pd.DataFrame(columns=cols)
    tot, abs_err, act, signed, wmape, mae, bias, cover = res
    return pd.DataFrame(
        [(pdf["material"].iloc[-1], pdf["plant"].iloc[-1], pdf["channel"].iloc[-1],
          tot, abs_err, act, signed, wmape, mae, bias, cover, MODEL)],
        columns=cols,
    )


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ACCURACY_TABLE} (
  forecast_version string, model_used string,
  material string, plant string, channel string,
  points_scored int, abs_err_sum double, act_sum double, signed_sum double,
  wmape double, mae double, bias double, pi_coverage double,
  granularity string, horizon int, computed_at timestamp
) USING DELTA
COMMENT 'Rolling-origin backtest accuracy per series for each demand-forecast version.'
""")

if BACKTEST_WINDOWS > 0:
    accuracy_df = (
        spark.table("history")
        .groupBy("material", "plant", "channel")
        .applyInPandas(backtest_group, schema=BT_SCHEMA)
        .withColumn("forecast_version", F.lit(FORECAST_VERSION))
        .withColumn("granularity", F.lit(GRANULARITY))
        .withColumn("horizon", F.lit(HORIZON))
        .withColumn("computed_at", F.current_timestamp())
        .selectExpr("forecast_version", "model_used", "material", "plant", "channel",
                    "points_scored", "abs_err_sum", "act_sum", "signed_sum",
                    "wmape", "mae", "bias", "pi_coverage", "granularity", "horizon", "computed_at")
    )
    accuracy_df.write.mode("append").saveAsTable(ACCURACY_TABLE)
    acc = spark.table(ACCURACY_TABLE).where(F.col("forecast_version") == FORECAST_VERSION)
    if acc.count():
        o = acc.selectExpr(
            "sum(abs_err_sum)/sum(act_sum) AS wmape",
            "sum(signed_sum)/sum(act_sum) AS bias",
            "sum(pi_coverage*points_scored)/sum(points_scored) AS coverage",
            "sum(points_scored) AS points",
            "count(1) AS series",
        ).collect()[0]
        print(f"Backtest [{MODEL}] series={o['series']} points={o['points']}  "
              f"WMAPE={o['wmape']:.3f}  bias={o['bias']:+.3f}  "
              f"PI_coverage={o['coverage']:.3f} (target ~{CONF:.2f})")
        display(acc.orderBy(F.col("wmape").desc()))
    else:
        print("Backtest produced no scored series (need >= MIN_TRAIN history + a test block).")
else:
    print("Backtest disabled (backtest_windows = 0).")

## Run the forecast across all series (distributed)

In [0]:
from datetime import timedelta

PERIOD_DELTA = {"D": timedelta(days=1), "W": timedelta(weeks=1), "M": timedelta(days=30)}[GRANULARITY]

OUTPUT_SCHEMA = (
    "mandt string, material string, plant string, channel string, uom string, "
    "horizon_step int, period_start date, forecast_qty double, "
    "lower_qty double, upper_qty double, std_qty double"
)

ORIGIN_TS = pd.Timestamp(forecast_origin)  # all series forecast forward from the one origin


def forecast_group(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("period_start")
    y = pdf["qty"].astype(float).to_numpy()
    mandt = pdf["mandt"].iloc[-1]; material = pdf["material"].iloc[-1]
    plant = pdf["plant"].iloc[-1]; channel = pdf["channel"].iloc[-1]
    uom = pdf["uom"].iloc[-1]
    mean, lo, hi = model_forecast(y, HORIZON, SEASON, MODEL, Z, LEVEL_PCT)
    rows = []
    for step in range(1, HORIZON + 1):
        m = max(float(mean[step - 1]), 0.0)
        l = min(max(float(lo[step - 1]), 0.0), m)         # lower <= forecast, >= 0
        u = max(float(hi[step - 1]), m)                    # upper >= forecast
        std = max((u - m) / Z, 0.0) if Z else 0.0
        ps = (ORIGIN_TS + PERIOD_DELTA * step).date()      # PeriodStart > ForecastOrigin
        rows.append((mandt, material, plant, channel, uom, step, ps, m, l, u, std))
    return pd.DataFrame(rows, columns=[c.split()[0] for c in OUTPUT_SCHEMA.split(", ")])


raw_fc = (
    spark.table("history")
    .groupBy("material", "plant", "channel")
    .applyInPandas(forecast_group, schema=OUTPUT_SCHEMA)
)
series_count = spark.table("history").select("material", "plant", "channel").distinct().count()
print(f"Series to forecast: {series_count:,}")

## Shape into the canonical `demandplan` schema and write

In [0]:
import json

final = (
    raw_fc
    .withColumn("Client_MANDT", F.col("mandt"))
    .withColumn("MaterialNumber_MATNR", F.col("material"))
    .withColumn("Plant_WERKS", F.col("plant"))
    .withColumn("DistributionChannel_VTWEG", F.col("channel"))
    .withColumn("Granularity", F.lit(GRANULARITY))
    .withColumn("PeriodStart", F.col("period_start"))
    .withColumn("ForecastVersion", F.lit(FORECAST_VERSION))
    .withColumn("HorizonStep", F.col("horizon_step"))
    .withColumn("ForecastOrigin", F.lit(forecast_origin).cast("date"))
    .withColumn("DemandDateBasis", F.lit(DEMAND_BASIS))
    .withColumn("ForecastQuantity", F.col("forecast_qty").cast("decimal(18,3)"))
    .withColumn("ForecastLowerQuantity", F.col("lower_qty").cast("decimal(18,3)"))
    .withColumn("ForecastUpperQuantity", F.col("upper_qty").cast("decimal(18,3)"))
    .withColumn("ForecastStdDevQuantity", F.col("std_qty").cast("decimal(18,3)"))
    .withColumn("ConfidenceLevel", F.lit(CONF).cast("decimal(4,3)"))
    .withColumn("UoM", F.col("uom"))
    .withColumn("DemandType", F.lit("INDEPENDENT"))
    .withColumn("ForecastModel", F.lit(MODEL))
    .withColumn("ModelParameters", F.lit(json.dumps(
        {"model": MODEL, "season": SEASON, "z": Z, "history_buckets": HISTORY,
         "backtest_windows": BACKTEST_WINDOWS})))
    .withColumn("GeneratedAt", F.current_timestamp())
    .withColumn("JobRunId", F.lit(FORECAST_RUN_ID))
    .withColumn("IsCurrentVersion", F.lit(True))
    .select(
        "Client_MANDT", "MaterialNumber_MATNR", "Plant_WERKS", "DistributionChannel_VTWEG",
        "Granularity", "PeriodStart", "ForecastVersion", "HorizonStep", "ForecastOrigin",
        "DemandDateBasis", "ForecastQuantity", "ForecastLowerQuantity", "ForecastUpperQuantity",
        "ForecastStdDevQuantity", "ConfidenceLevel", "UoM", "DemandType", "ForecastModel",
        "ModelParameters", "GeneratedAt", "JobRunId", "IsCurrentVersion",
    )
)

# gold.demandplan already exists (canonical shape); IF NOT EXISTS is a safe no-op elsewhere.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OUT_TABLE} (
  Client_MANDT string, MaterialNumber_MATNR string, Plant_WERKS string,
  DistributionChannel_VTWEG string, Granularity string, PeriodStart date,
  ForecastVersion string, HorizonStep int, ForecastOrigin date, DemandDateBasis string,
  ForecastQuantity decimal(18,3), ForecastLowerQuantity decimal(18,3),
  ForecastUpperQuantity decimal(18,3), ForecastStdDevQuantity decimal(18,3),
  ConfidenceLevel decimal(4,3), UoM string, DemandType string, ForecastModel string,
  ModelParameters string, GeneratedAt timestamp, JobRunId string, IsCurrentVersion boolean
) USING DELTA
COMMENT 'Independent-demand forecast (canonical demandplan shape). Consumed by the Demand Twin and planning.'
""")

# Demote previous current version, then append the new one.
spark.sql(f"UPDATE {OUT_TABLE} SET IsCurrentVersion = false WHERE IsCurrentVersion = true")
final.write.mode("append").saveAsTable(OUT_TABLE)
rows_written = spark.table(OUT_TABLE).where(F.col("ForecastVersion") == FORECAST_VERSION).count()
print(f"Wrote {rows_written:,} forecast rows as version {FORECAST_VERSION} to {OUT_TABLE}")

## Verify

In [0]:
display(spark.sql(f"""
SELECT MaterialNumber_MATNR, Plant_WERKS, DistributionChannel_VTWEG,
       PeriodStart, HorizonStep, ForecastQuantity, ForecastLowerQuantity, ForecastUpperQuantity
FROM {OUT_TABLE}
WHERE IsCurrentVersion = true
ORDER BY MaterialNumber_MATNR, Plant_WERKS, DistributionChannel_VTWEG, PeriodStart
LIMIT 50
"""))

## Next
The forecast now populates `gold.demandplan` (with backtest accuracy in
`monitoring.demand_forecast_accuracy`). Proceed to build the **Demand Twin**, which combines
this forecast with firm/open demand and backlog from sales orders, and surfaces
forecast-accuracy/bias signals. See the orchestrator skill, Step 4.